## Cell 1 — Instalasi Dependensi

In [ ]:
# Install library yang dibutuhkan
!pip install playwright pandas --quiet

# Install browser Chromium untuk Playwright
!playwright install chromium

print(" Instalasi selesai!")

(node:1422) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 118.0s167.3 MiB [] 0% 378.7s167.3 MiB [] 0% 434.9s167.3 MiB [] 0% 348.7s167.3 MiB [] 0% 300.9s167.3 MiB [] 0% 271.4s167.3 MiB [] 0% 250.2s167.3 MiB [] 0% 234.6s167.3 MiB [] 0% 219.5s167.3 MiB [] 0% 206.5s167.3 MiB [] 0% 194.9s167.3 MiB [] 0% 171.0s167.3 MiB [] 0% 153.9s167.3 MiB [] 0% 141.1s167.3 MiB [] 0% 131.0s167.3 MiB [] 0% 122.6s167.3 MiB [] 0% 113.1s167.3 MiB [] 0% 103.5s167.3 MiB [] 0% 94.0s167.3 MiB [] 0% 84.7s167.3 MiB [] 0% 76.9s167.3 MiB [] 0% 70.9s167.3 MiB [] 0% 65.2s167.3 MiB [] 1% 59.0s167.3 MiB [] 1% 54.0s167.3 MiB [] 1% 48.8s167.3 MiB [] 1% 44.2s167.3 MiB [] 1% 39.9s167.3 MiB [] 1% 36.2s167.3 MiB [] 2% 32.6s167.3 MiB [] 2% 29.4s167.3 MiB [

In [ ]:
# Install additional system libraries required by Chromium in Colab
!apt-get update && apt-get install -y libatk-adaptor libatk-bridge2.0-0 libgtk-3-0 libgbm-dev libu2f-udev xvfb

print("Additional Chromium dependencies installation attempt complete.")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,192 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd6

## Cell 2 — Import Library

In [ ]:
import time
import random
import pandas as pd
from playwright.async_api import async_playwright # Changed to async_playwright
import asyncio # Import asyncio for async operations
import os

print("Semua library berhasil diimport!")

✅ Semua library berhasil diimport!


## Cell 3 — Konfigurasi Selector CSS

In [ ]:
# Selector kandidat – dicoba satu per satu, ambil yang menghasilkan teks
ANSWER_SELECTORS = [
    "div.q-box.qu-userSelect--text span",          # struktur lama
    "div[class*='q-text'] span",                   # kelas dinamis
    "div.puppeteer_test_answer_content span",       # kelas puppeteer
    "div[data-testid='answer'] span",              # data-testid baru
    "div.AnswerBase div.q-box span",               # wrapper jawaban
    "div.spacing_log_answer_content span",         # spacing wrapper
    "span.q-box.qu-userSelect--text",              # span langsung
    "div[class*='AnswerPagedList'] span",           # paged list
    "div[class*='answer'] p",                      # paragraph fallback
    "article span",                                # artikel generik
]

print(f"✅ {len(ANSWER_SELECTORS)} selector CSS siap digunakan.")

✅ 10 selector CSS siap digunakan.


## Cell 4 — Fungsi Helper: `_collect_texts_from_page()`

Fungsi ini mencoba semua selector CSS dan mengembalikan daftar teks yang ditemukan di halaman aktif.

In [5]:
async def _collect_texts_from_page(page) -> list:
    """
    Coba semua selector dan kembalikan daftar teks unik yang ditemukan.
    Hanya teks dengan panjang > 30 karakter yang diambil.
    """
    found = []
    for sel in ANSWER_SELECTORS:
        try:
            # Await Playwright calls
            elements = await page.query_selector_all(sel)
            for el in elements:
                txt = (await el.inner_text()).strip()
                if len(txt) > 30:          # buang teks terlalu pendek
                    found.append(txt)
        except Exception:
            continue
    return found

print("✅ Fungsi helper `_collect_texts_from_page` siap.")

✅ Fungsi helper `_collect_texts_from_page` siap.


## Cell 5 — Fungsi Utama: `scrape_quora_posts()`

In [ ]:
async def scrape_quora_posts(urls: list, target_per_page: int = 500) -> None:
    all_records = []

    # Changed to async_playwright
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,                          # Change to True for Colab environment
            args=["--start-maximized"],
        )

        context = await browser.new_context(
            viewport={"width": 1280, "height": 720},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
        )

        for url_idx, url in enumerate(urls, start=1):
            print(f"\n{'='*60}")
            print(f"[{url_idx}/{len(urls)}] Membuka: {url}")
            print("="*60)

            page = await context.new_page()

            # ── Buka halaman ──────────────────────────────────────────
            try:
                await page.goto(url, timeout=60_000, wait_until="domcontentloaded") # Await goto
            except Exception as e:
                print(f"  [ERROR] Gagal membuka halaman: {e}")
                await page.close()
                continue

            # ── Jeda login / popup (20–30 detik) ─────────────────────
            wait_sec = random.randint(20, 30)
            print(f"  [WAIT] Jeda {wait_sec} detik – silakan login atau tutup popup jika perlu …")
            await asyncio.sleep(wait_sec) # Changed to asyncio.sleep

            # ── Auto-scroll scraping ────────────────────────────────
            page_texts   = set()                  # set = otomatis deduplikasi
            max_scrolls  = random.randint(120, 150)
            scroll_step  = 1000                   # px per scroll
            no_new_count = 0                      # counter: scroll tanpa data baru
            NO_NEW_LIMIT = 15                     # berhenti jika 15 scroll tanpa data

            print(f"  [INFO] Target: {target_per_page} teks | Maks scroll: {max_scrolls}")

            for scroll_i in range(1, max_scrolls + 1):

                # ── Scroll ────────────────────────────────────────────
                try:
                    await page.evaluate(f"window.scrollBy(0, {scroll_step})") # Await evaluate
                except Exception as e:
                    print(f"  [WARN] Scroll error: {e}")
                    break

                # ── Kumpulkan teks ────────────────────────────────────
                try:
                    new_texts = await _collect_texts_from_page(page) # Await helper function
                    before    = len(page_texts)
                    page_texts.update(new_texts)
                    gained    = len(page_texts) - before

                    if gained == 0:
                        no_new_count += 1
                    else:
                        no_new_count = 0

                    # Progres setiap 10 scroll
                    if scroll_i % 10 == 0:
                        print(
                            f"  [SCROLL {scroll_i:>3}] "
                            f"Teks terkumpul: {len(page_texts):>4} / {target_per_page}"
                        )

                except Exception as e:
                    print(f"  [WARN] Scraping error di scroll {scroll_i}: {e}")

                # ── Kondisi berhenti ──────────────────────────────────
                if len(page_texts) >= target_per_page:
                    print(f"  [DONE] Target {target_per_page} tercapai pada scroll ke-{scroll_i}.")
                    break

                if no_new_count >= NO_NEW_LIMIT:
                    print(f"  [DONE] Tidak ada data baru selama {NO_NEW_LIMIT} scroll berturut-turut.")
                    break

                # ── Jeda acak antar scroll ────────────────────────────
                await asyncio.sleep(random.uniform(2.0, 4.0)) # Changed to asyncio.sleep

            # ── Simpan ke list utama ──────────────────────────────────
            collected = list(page_texts)[:target_per_page]   # batasi sesuai target
            print(f"  [RESULT] Total teks dari halaman ini: {len(collected)}")

            for text in collected:
                all_records.append({
                    "source"  : "Quora",
                    "app_name": "E-Wallet Discussion",
                    "content" : text,
                    "score"   : None,
                })

            await page.close()

        await browser.close()

    # ── Simpan ke CSV ─────────────────────────────────────────────────
    if all_records:
        df = pd.DataFrame(all_records)
        output_file = "dataset_quora_ewallet.csv"
        df.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"\n[SAVED] {len(df)} baris data disimpan ke '{output_file}'")
        print("\nPreview 5 baris pertama:")
        display(df.head())
    else:
        print("\n[WARNING] Tidak ada data yang berhasil dikumpulkan.")

print("Fungsi `scrape_quora_posts` siap digunakan.")

✅ Fungsi `scrape_quora_posts` siap digunakan.


## Cell 6 — Daftar URL Target

Berisi URL pertanyaan spesifik dan halaman topik Quora yang relevan dengan **E-Wallet**.

In [ ]:
quora_urls = [
    # ── Pertanyaan spesifik ──────────────────────────────────────────────
    "https://www.quora.com/What-is-the-best-e-wallet-in-Southeast-Asia",
    "https://www.quora.com/Which-e-wallet-is-the-safest-to-use",
    "https://www.quora.com/What-are-the-advantages-of-using-an-e-wallet",
    "https://www.quora.com/How-do-e-wallets-work",
    "https://www.quora.com/Is-GoPay-or-OVO-better-in-Indonesia",
    "https://www.quora.com/What-is-the-difference-between-GoPay-OVO-and-Dana",
    "https://www.quora.com/How-secure-are-digital-wallets",
    "https://www.quora.com/What-are-the-best-e-wallets-available-in-Indonesia",

    # ── Topik / tag ──────────────────────────────────────────────────────
    "https://www.quora.com/topic/E-Wallets",
    "https://www.quora.com/topic/Digital-Wallets",
    "https://www.quora.com/topic/Mobile-Payments",
    "https://www.quora.com/topic/GoPay",
    "https://www.quora.com/topic/OVO-Indonesia",
    "https://www.quora.com/topic/Dana-e-wallet",
]

print(f"Total URL yang akan di-scrape: {len(quora_urls)}")
for i, u in enumerate(quora_urls, 1):
    print(f"  {i:>2}. {u}")

✅ Total URL yang akan di-scrape: 14
   1. https://www.quora.com/What-is-the-best-e-wallet-in-Southeast-Asia
   2. https://www.quora.com/Which-e-wallet-is-the-safest-to-use
   3. https://www.quora.com/What-are-the-advantages-of-using-an-e-wallet
   4. https://www.quora.com/How-do-e-wallets-work
   5. https://www.quora.com/Is-GoPay-or-OVO-better-in-Indonesia
   6. https://www.quora.com/What-is-the-difference-between-GoPay-OVO-and-Dana
   7. https://www.quora.com/How-secure-are-digital-wallets
   8. https://www.quora.com/What-are-the-best-e-wallets-available-in-Indonesia
   9. https://www.quora.com/topic/E-Wallets
  10. https://www.quora.com/topic/Digital-Wallets
  11. https://www.quora.com/topic/Mobile-Payments
  12. https://www.quora.com/topic/GoPay
  13. https://www.quora.com/topic/OVO-Indonesia
  14. https://www.quora.com/topic/Dana-e-wallet


## Cell 7 — Jalankan Scraper

In [ ]:
# Konfigurasi
TARGET_PER_PAGE = 1000   # jumlah teks per URL (turunkan ke 200 untuk testing)

# Jalankan scraper
# Use await to execute the async function within Colab's existing event loop
await scrape_quora_posts(urls=quora_urls, target_per_page=TARGET_PER_PAGE)


[1/14] Membuka: https://www.quora.com/What-is-the-best-e-wallet-in-Southeast-Asia
  [WAIT] Jeda 20 detik – silakan login atau tutup popup jika perlu …
  [INFO] Target: 1000 teks | Maks scroll: 139
  [SCROLL  10] Teks terkumpul:    0 / 1000
  [DONE] Tidak ada data baru selama 15 scroll berturut-turut.
  [RESULT] Total teks dari halaman ini: 0

[2/14] Membuka: https://www.quora.com/Which-e-wallet-is-the-safest-to-use
  [WAIT] Jeda 24 detik – silakan login atau tutup popup jika perlu …
  [INFO] Target: 1000 teks | Maks scroll: 121
  [SCROLL  10] Teks terkumpul:    0 / 1000
  [DONE] Tidak ada data baru selama 15 scroll berturut-turut.
  [RESULT] Total teks dari halaman ini: 0

[3/14] Membuka: https://www.quora.com/What-are-the-advantages-of-using-an-e-wallet
  [WAIT] Jeda 30 detik – silakan login atau tutup popup jika perlu …
  [INFO] Target: 1000 teks | Maks scroll: 132
  [SCROLL  10] Teks terkumpul:    0 / 1000
  [DONE] Tidak ada data baru selama 15 scroll berturut-turut.
  [RESULT] Tot

,source,app_name,content,score
0,Quora,E-Wallet Discussion,Digital wallets are categorized,None
1,Quora,E-Wallet Discussion,The wallet when sending cryptocurrency will as...,None
2,Quora,E-Wallet Discussion,Author has 346 answers and 5.3M answer views,None
3,Quora,E-Wallet Discussion,It’s a technology that allows you to keep and ...,None
4,Quora,E-Wallet Discussion,What types of things can I store in an e-wallet?,None


## Cell 8 — Preview & Analisis Hasil

Jalankan cell ini setelah scraping selesai untuk melihat ringkasan data yang terkumpul.

In [ ]:
output_file = "dataset_quora_ewallet.csv"

if os.path.exists(output_file):
    df = pd.read_csv(output_file)

    print("=" * 50)
    print("RINGKASAN DATASET")
    print("=" * 50)
    print(f"  Total baris   : {len(df):,}")
    print(f"  Total kolom   : {len(df.columns)}")
    print(f"  Kolom         : {list(df.columns)}")
    print(f"  Duplikat      : {df.duplicated(subset='content').sum()}")
    print(f"  Missing value : {df['content'].isna().sum()}")
    print(f"  Rata-rata panjang teks: {df['content'].str.len().mean():.0f} karakter")
    print()

    print("5 Baris Pertama:")
    display(df.head())

    print("\n5 Baris Terakhir:")
    display(df.tail())

    print("\nDistribusi panjang teks (karakter):")
    display(df['content'].str.len().describe().rename("panjang_teks").to_frame())
else:
    print(f"File '{output_file}' belum ditemukan. Jalankan Cell 7 terlebih dahulu.")

📊 RINGKASAN DATASET
  Total baris   : 984
  Total kolom   : 4
  Kolom         : ['source', 'app_name', 'content', 'score']
  Duplikat      : 40
  Missing value : 0
  Rata-rata panjang teks: 234 karakter

📋 5 Baris Pertama:


,source,app_name,content,score
0,Quora,E-Wallet Discussion,Digital wallets are categorized,NaN
1,Quora,E-Wallet Discussion,The wallet when sending cryptocurrency will as...,NaN
2,Quora,E-Wallet Discussion,Author has 346 answers and 5.3M answer views,NaN
3,Quora,E-Wallet Discussion,It’s a technology that allows you to keep and ...,NaN
4,Quora,E-Wallet Discussion,What types of things can I store in an e-wallet?,NaN



📋 5 Baris Terakhir:


,source,app_name,content,score
979,Quora,E-Wallet Discussion,I am not wiriting the fact you like but what I...,NaN
980,Quora,E-Wallet Discussion,What does “Selamat datang di gopay” mean?,NaN
981,Quora,E-Wallet Discussion,What do you think about PayPal becoming the fi...,NaN
982,Quora,E-Wallet Discussion,Doesn’t matter. PayPal is and never was a big ...,NaN
983,Quora,E-Wallet Discussion,AboutCareersPrivacyTermsContactLanguagesYour A...,NaN



📈 Distribusi panjang teks (karakter):


,panjang_teks
count,984.000000
mean,234.041667
std,403.299234
min,31.000000
25%,49.000000
50%,109.500000
75%,300.000000
max,6618.000000
